# Nav2 Benchmark — Experiment Driver

Self-contained notebook that drives the Nav2 controller benchmark from inside
Jupyter. It launches the simulation, runs hard-coded trials with widgets,
appends metrics to a CSV, and analyses the collected results.

Ported from `scripts/simple_driver.py` and the launch command:

```
ros2 launch nav2_benchmark benchmark.launch.py \
    world:=$(ros2 pkg prefix --share nav2_benchmark)/worlds/corridor.world \
    map:=$(ros2 pkg prefix --share nav2_benchmark)/maps/corridor.yaml \
    controller:=<dwb|mppi|rpp>
ros2 run nav2_benchmark dynamic_obstacles.py --ros-args -p use_sim_time:=true
```


## 1. Setup — launch helpers and imports

In [ ]:
import csv
import math
import os
import signal
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import ipywidgets as widgets
from IPython.display import display

WORKSPACE_SETUP = os.path.expanduser('~/research_ws/install/local_setup.bash')
PKG = 'nav2_benchmark'


def _bash_popen(cmd: str):
    return subprocess.Popen(
        f'source {WORKSPACE_SETUP} && {cmd}',
        shell=True,
        executable='/bin/bash',
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        text=True,
        env=os.environ,
        preexec_fn=os.setsid,
    )


def _pkg_share() -> str:
    out = subprocess.check_output(
        ['bash', '-lc',
         f'source {WORKSPACE_SETUP} && ros2 pkg prefix --share {PKG}'],
        text=True,
    ).strip()
    return out


def start_ros2_launch(controller: str):
    share = _pkg_share()
    world = f'{share}/worlds/corridor.world'
    map_yaml = f'{share}/maps/corridor.yaml'
    cmd = (
        f'ros2 launch {PKG} benchmark.launch.py '
        f'world:={world} map:={map_yaml} controller:={controller}'
    )
    return _bash_popen(cmd)


def start_dynamic_obstacles():
    return _bash_popen(
        f'ros2 run {PKG} dynamic_obstacles.py --ros-args -p use_sim_time:=true'
    )


def stop_ros2_launch(process):
    if process is None:
        return
    if process.poll() is not None:
        return
    try:
        pgid = os.getpgid(process.pid)
    except ProcessLookupError:
        return
    try:
        os.killpg(pgid, signal.SIGINT)
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    except ProcessLookupError:
        pass


## 2. Simulation control widgets

In [ ]:
controller_dd = widgets.Dropdown(
    options=['dwb', 'mppi', 'rpp'], value='mppi',
    description='Controller:')
launch_btn = widgets.Button(description='Launch Simulation', button_style='success')
stop_btn   = widgets.Button(description='Stop Simulation',   button_style='danger')
sim_status = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '4px'})

_procs = {'launch': None, 'obstacles': None}


def _on_launch(b):
    with sim_status:
        sim_status.clear_output(wait=True)
        if _procs['launch'] is not None and _procs['launch'].poll() is None:
            print('Simulation already running. Stop it first.')
            return
        ctrl = controller_dd.value
        print(f'Launching benchmark.launch.py controller={ctrl} ...')
        _procs['launch'] = start_ros2_launch(ctrl)
        time.sleep(3.0)
        print('Starting dynamic_obstacles node ...')
        _procs['obstacles'] = start_dynamic_obstacles()
        print('Sim launched. Wait ~15 s for Nav2 / Gazebo to come up '
              'before running the experiment.')


def _on_stop(b):
    with sim_status:
        sim_status.clear_output(wait=True)
        print('Stopping dynamic_obstacles ...')
        stop_ros2_launch(_procs['obstacles']); _procs['obstacles'] = None
        print('Stopping benchmark.launch.py ...')
        stop_ros2_launch(_procs['launch']);    _procs['launch']    = None
        print('All processes stopped.')


launch_btn.on_click(_on_launch)
stop_btn.on_click(_on_stop)

display(widgets.HBox([controller_dd, launch_btn, stop_btn]), sim_status)


## 3. Experiment configuration widgets

In [ ]:
condition_dd  = widgets.Dropdown(options=['static', 'dynamic'], value='dynamic',
                                description='Condition:')
n_trials_sl   = widgets.IntSlider(value=5,  min=1,  max=25,  description='Trials:')
timeout_sl    = widgets.IntSlider(value=90, min=30, max=120, description='Timeout (s):')
output_csv_tx = widgets.Text(value='results/run.csv', description='Output CSV:')

display(widgets.VBox([condition_dd, n_trials_sl, timeout_sl, output_csv_tx]))


## 4. Hardcoded trial list

Each entry is `(start_pose, goal_pose)`, where a pose is `(x, y, yaw_radians)`.

In [ ]:
TRIALS = [
    ((1.0,  5.0,  0.0),  (0.0,  0.0, 0.0)),
    ((-2.0, -1.0, 1.57), (2.0,  1.5, 0.0)),
    ((0.0,  0.0,  0.0),  (-4.0, -1.0, 0.0)),
    ((0.0,  0.0,  0.0),  (-4.0, -1.0, 0.0)),
    ((1.0,  5.0,  0.0),  (0.0,  0.0, 0.0)),
]


## 5. Experiment driver

`SimpleDriver` and `compute_metrics` ported from `scripts/simple_driver.py`.
CSV columns are kept byte-for-byte identical.

In [ ]:
import rclpy
from rclpy.executors import SingleThreadedExecutor
from rclpy.node import Node

from geometry_msgs.msg import PoseStamped, PoseWithCovarianceStamped
from gazebo_msgs.msg import ModelStates
from gazebo_msgs.srv import SetEntityState
from std_srvs.srv import Trigger

from nav2_simple_commander.robot_navigator import BasicNavigator, TaskResult

ENTITY_NAMES             = ['waffle', 'turtlebot3_waffle']
TELEPORT_SERVICES        = ['/gazebo/set_entity_state', '/set_entity_state']
CYLINDER_NAME_PREFIXES   = ('moving_cylinder_', 'cyl_')
MODEL_STATES_TOPICS      = ('/model_states', '/gazebo/model_states')
SAMPLE_PERIOD            = 0.1
CLOSE_APPROACH_THRESHOLD = 0.30

CSV_COLUMNS = [
    'run_id', 'trial_idx',
    'start_x', 'start_y', 'start_yaw',
    'goal_x',  'goal_y',  'goal_yaw',
    'success', 'timeout',
    'time_to_goal', 'path_length', 'path_efficiency',
    'straight_line', 'min_clearance', 'mean_jerk', 'n_close_approach',
]


def yaw_to_quat_zw(yaw):
    return math.sin(yaw / 2.0), math.cos(yaw / 2.0)


def quat_to_yaw(q):
    return math.atan2(2.0 * (q.w * q.z + q.x * q.y),
                      1.0 - 2.0 * (q.y * q.y + q.z * q.z))


def make_pose_stamped(x, y, yaw, stamp, frame_id='map'):
    p = PoseStamped()
    p.header.frame_id = frame_id
    p.header.stamp = stamp
    p.pose.position.x = float(x)
    p.pose.position.y = float(y)
    qz, qw = yaw_to_quat_zw(yaw)
    p.pose.orientation.z = qz
    p.pose.orientation.w = qw
    return p


class SimpleDriver(Node):
    def __init__(self):
        super().__init__('notebook_simple_driver')

        self.start_cli = self.create_client(Trigger, '/obstacles/start')
        self.stop_cli  = self.create_client(Trigger, '/obstacles/stop')
        for cli, name in [(self.start_cli, '/obstacles/start'),
                          (self.stop_cli,  '/obstacles/stop')]:
            while not cli.wait_for_service(timeout_sec=2.0):
                self.get_logger().info(f'Waiting for {name} ...')

        self.gz_cli = None
        for srv in TELEPORT_SERVICES:
            cli = self.create_client(SetEntityState, srv)
            if cli.wait_for_service(timeout_sec=2.0):
                self.gz_cli = cli
                self.get_logger().info(f'Using {srv} for teleport.')
                break
        if self.gz_cli is None:
            raise RuntimeError(f'None of {TELEPORT_SERVICES} available.')

        self.initial_pose_pub = self.create_publisher(
            PoseWithCovarianceStamped, '/initialpose', 10)

        self.latest_states = None
        for topic in MODEL_STATES_TOPICS:
            self.create_subscription(
                ModelStates, topic, self._on_model_states, 10)

        self._executor = SingleThreadedExecutor()
        self._executor.add_node(self)
        self._spinning = True
        self._spin_thread = threading.Thread(target=self._spin_loop, daemon=True)
        self._spin_thread.start()

        deadline = time.monotonic() + 5.0
        while self.latest_states is None and time.monotonic() < deadline:
            time.sleep(0.1)
        if self.latest_states is None:
            self.get_logger().warn(
                f'No ModelStates on any of {MODEL_STATES_TOPICS} within 5 s.')

    def _spin_loop(self):
        while rclpy.ok() and self._spinning:
            try:
                self._executor.spin_once(timeout_sec=0.1)
            except Exception:
                break

    def shutdown(self):
        self._spinning = False
        try:
            self._executor.remove_node(self)
        except Exception:
            pass

    def _on_model_states(self, msg):
        self.latest_states = msg

    def _wait_for_future(self, fut, timeout=5.0):
        event = threading.Event()
        fut.add_done_callback(lambda _f: event.set())
        return fut.result() if event.wait(timeout) else None

    def publish_initial_pose(self, x, y, yaw, xy_var=0.05, yaw_var=0.05):
        msg = PoseWithCovarianceStamped()
        msg.header.frame_id = 'map'
        msg.header.stamp = self.get_clock().now().to_msg()
        msg.pose.pose.position.x = float(x)
        msg.pose.pose.position.y = float(y)
        qz, qw = yaw_to_quat_zw(yaw)
        msg.pose.pose.orientation.z = qz
        msg.pose.pose.orientation.w = qw
        cov = [0.0] * 36
        cov[0]  = xy_var
        cov[7]  = xy_var
        cov[35] = yaw_var
        msg.pose.covariance = cov
        self.initial_pose_pub.publish(msg)

    def call_trigger(self, cli):
        fut = cli.call_async(Trigger.Request())
        return self._wait_for_future(fut)

    def teleport(self, x, y, yaw):
        qz, qw = yaw_to_quat_zw(yaw)
        for entity_name in ENTITY_NAMES:
            req = SetEntityState.Request()
            req.state.name = entity_name
            req.state.pose.position.x = float(x)
            req.state.pose.position.y = float(y)
            req.state.pose.position.z = 0.0
            req.state.pose.orientation.z = qz
            req.state.pose.orientation.w = qw
            req.state.reference_frame = 'world'
            fut = self.gz_cli.call_async(req)
            res = self._wait_for_future(fut)
            if res is not None and res.success:
                return True
        return False

    def _sample_from_states(self, t_elapsed):
        msg = self.latest_states
        if msg is None:
            return None
        robot_idx = None
        for name in ENTITY_NAMES:
            if name in msg.name:
                robot_idx = msg.name.index(name)
                break
        if robot_idx is None:
            return None
        pose  = msg.pose[robot_idx]
        twist = msg.twist[robot_idx]
        x = pose.position.x
        y = pose.position.y
        yaw = quat_to_yaw(pose.orientation)
        v = math.hypot(twist.linear.x, twist.linear.y)
        cyl_xys = [
            (msg.pose[i].position.x, msg.pose[i].position.y)
            for i, n in enumerate(msg.name)
            if any(n.startswith(p) for p in CYLINDER_NAME_PREFIXES)
        ]
        if cyl_xys:
            min_clearance = min(math.hypot(x - cx, y - cy) for cx, cy in cyl_xys)
        else:
            min_clearance = float('inf')
        return (t_elapsed, x, y, yaw, v, min_clearance)

    def collect_trial(self, nav, timeout_sec):
        trial_start = nav.get_clock().now()
        log = []
        next_sample_at = 0.0
        timeout_fired = False
        while True:
            now = nav.get_clock().now()
            elapsed = (now - trial_start).nanoseconds / 1e9
            if nav.isTaskComplete():
                break
            if elapsed > timeout_sec:
                self.get_logger().warn(
                    f'Trial timeout ({timeout_sec:.0f} s) — cancelling.')
                nav.cancelTask()
                timeout_fired = True
                while not nav.isTaskComplete():
                    pass
                break
            if elapsed >= next_sample_at:
                sample = self._sample_from_states(elapsed)
                if sample is not None and (not log or sample[0] > log[-1][0]):
                    log.append(sample)
                next_sample_at += SAMPLE_PERIOD
        return log, timeout_fired


RESULT_LABEL = {
    TaskResult.SUCCEEDED: 'SUCCEEDED',
    TaskResult.CANCELED:  'CANCELED',
    TaskResult.FAILED:    'FAILED',
}


def compute_metrics(log, start_xy, goal_xy):
    straight_line = math.hypot(goal_xy[0] - start_xy[0],
                               goal_xy[1] - start_xy[1])
    if len(log) == 0:
        return {
            'time_to_goal': 0.0,
            'path_length': 0.0,
            'path_efficiency': 0.0,
            'straight_line': straight_line,
            'min_clearance': float('inf'),
            'mean_jerk': 0.0,
            'n_close_approach': 0,
        }
    t  = np.array([s[0] for s in log], dtype=float)
    xs = np.array([s[1] for s in log], dtype=float)
    ys = np.array([s[2] for s in log], dtype=float)
    v  = np.array([s[4] for s in log], dtype=float)
    cl = [s[5] for s in log]
    time_to_goal = float(t[-1])
    path_length  = float(np.sum(np.hypot(np.diff(xs), np.diff(ys))))
    path_efficiency = (straight_line / path_length) if path_length > 0 else 0.0
    finite_cl = [c for c in cl if c is not None and not math.isinf(c)]
    min_clearance = float(min(finite_cl)) if finite_cl else float('inf')
    if len(t) < 4:
        mean_jerk = 0.0
    else:
        keep = np.concatenate(([True], np.diff(t) > 0))
        t_u = t[keep]; v_u = v[keep]
        if len(t_u) < 4:
            mean_jerk = 0.0
        else:
            acc  = np.gradient(v_u, t_u)
            jerk = np.gradient(acc, t_u)
            mean_jerk = float(np.mean(np.abs(jerk)))
    n_close_approach = sum(
        1 for c in cl if c is not None and c < CLOSE_APPROACH_THRESHOLD)
    return {
        'time_to_goal': time_to_goal,
        'path_length': path_length,
        'path_efficiency': path_efficiency,
        'straight_line': straight_line,
        'min_clearance': min_clearance,
        'mean_jerk': mean_jerk,
        'n_close_approach': n_close_approach,
    }


def append_csv_row(path, row):
    parent = os.path.dirname(os.path.abspath(path))
    os.makedirs(parent, exist_ok=True)
    new_file = not os.path.exists(path)
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)


## 6. Run experiment

The trial loop runs in a daemon thread so the kernel stays responsive. Each
trial prints one line into the output widget below. `condition=static` skips
the `/obstacles/start` trigger; the cylinders stay parked in their phase-0
positions.

In [ ]:
run_btn = widgets.Button(description='Run Experiment', button_style='primary')
run_status = widgets.Output(layout={'border': '1px solid #ddd',
                                    'padding': '4px',
                                    'max_height': '320px',
                                    'overflow': 'auto'})

_run_thread = {'t': None}


def _trial_loop(controller, condition, n_trials, timeout_sec, output_csv, status):
    def log(msg):
        with status:
            print(msg)
    try:
        if not rclpy.ok():
            rclpy.init()
        log(f'Connecting to ROS (controller={controller}, condition={condition}) ...')
        driver = SimpleDriver()
        nav = BasicNavigator()
        nav.waitUntilNav2Active()
        log('Nav2 active. Starting trials.')

        run_id = f'{controller}_corridor_{condition}_{datetime.now():%Y%m%d_%H%M%S}'
        seq = [TRIALS[i % len(TRIALS)] for i in range(n_trials)]

        for i, (start, goal) in enumerate(seq, start=1):
            driver.call_trigger(driver.stop_cli)
            driver.teleport(*start)
            driver.publish_initial_pose(*start)
            nav.clearAllCostmaps(); time.sleep(0.5)
            nav.clearAllCostmaps(); time.sleep(0.5)
            nav.clearAllCostmaps()
            time.sleep(3.0)
            nav.clearAllCostmaps()
            time.sleep(2.0)

            if condition == 'dynamic':
                driver.call_trigger(driver.start_cli)

            goal_stamp = nav.get_clock().now().to_msg()
            nav.goToPose(make_pose_stamped(*goal, stamp=goal_stamp))

            log_pts, timeout_fired = driver.collect_trial(nav, timeout_sec)
            driver.call_trigger(driver.stop_cli)

            result = nav.getResult()
            success = (result == TaskResult.SUCCEEDED) and not timeout_fired
            metrics = compute_metrics(log_pts,
                                      (start[0], start[1]),
                                      (goal[0],  goal[1]))
            row = {
                'run_id':   run_id,  'trial_idx': i,
                'start_x':  start[0], 'start_y':  start[1], 'start_yaw': start[2],
                'goal_x':   goal[0],  'goal_y':   goal[1],  'goal_yaw':  goal[2],
                'success':  success,  'timeout':  timeout_fired,
                **metrics,
            }
            append_csv_row(output_csv, row)

            log(f'Trial {i}/{n_trials}: '
                f'result={RESULT_LABEL.get(result, str(result))} '
                f'success={success} timeout={timeout_fired} '
                f't={metrics["time_to_goal"]:.1f}s '
                f'len={metrics["path_length"]:.2f}m '
                f'eff={metrics["path_efficiency"]:.2f} '
                f'samples={len(log_pts)} -> {output_csv}')

            if i < n_trials:
                nav.clearAllCostmaps()
                time.sleep(2.0)

        log('All trials done.')
        driver.shutdown()
        driver.destroy_node()
    except Exception as e:
        log(f'ERROR: {e!r}')


def _on_run(b):
    with run_status:
        run_status.clear_output(wait=True)
    if _run_thread['t'] is not None and _run_thread['t'].is_alive():
        with run_status:
            print('Run already in progress — wait for it to finish.')
        return
    args = (
        controller_dd.value,
        condition_dd.value,
        int(n_trials_sl.value),
        float(timeout_sl.value),
        output_csv_tx.value,
        run_status,
    )
    _run_thread['t'] = threading.Thread(target=_trial_loop, args=args, daemon=True)
    _run_thread['t'].start()


run_btn.on_click(_on_run)
display(run_btn, run_status)


## 7. Analysis

These cells operate on the on-disk CSVs and do not require Nav2 / Gazebo to
be running. They look for files named `results/<controller>_corridor_<condition>.csv`
for `controller ∈ {dwb, mppi, rpp}` and `condition ∈ {static, dynamic}`.

In [ ]:
RESULTS_DIR = Path('results')
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

CONTROLLERS = ['dwb', 'mppi', 'rpp']
CONDITIONS  = ['static', 'dynamic']


def load_all() -> pd.DataFrame:
    frames = []
    for ctrl in CONTROLLERS:
        for cond in CONDITIONS:
            path = RESULTS_DIR / f'{ctrl}_corridor_{cond}.csv'
            if path.exists():
                df = pd.read_csv(path)
                df['controller'] = ctrl
                df['condition']  = cond
                frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


df_all = load_all()
print(f'Loaded {len(df_all)} rows from results/<ctrl>_corridor_<cond>.csv.')
df_all.head()


In [ ]:
def median_iqr(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return 'NA'
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return f'{s.median():.2f} [{q1:.2f}-{q3:.2f}]'


def summarise(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    g = df.groupby(['controller', 'condition'])
    out = pd.DataFrame({
        'n':                 g.size(),
        'success_rate':      g['success'].mean().round(3),
        'time_to_goal':      g['time_to_goal'].apply(median_iqr),
        'path_efficiency':   g['path_efficiency'].apply(median_iqr),
        'min_clearance':     g['min_clearance'].apply(median_iqr),
        'mean_jerk':         g['mean_jerk'].apply(median_iqr),
        'n_close_approach':  g['n_close_approach'].apply(median_iqr),
    })
    return out.reset_index()


summary = summarise(df_all)
summary


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100


def _clean(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    return df.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['time_to_goal', 'path_efficiency', 'min_clearance', 'mean_jerk']
    )


def _no_data(ax):
    ax.text(0.5, 0.5, 'no data', ha='center', va='center',
            transform=ax.transAxes)


def fig_success_rate(df):
    fig, ax = plt.subplots(figsize=(7, 4))
    if df.empty:
        _no_data(ax); ax.set_title('Success rate'); return fig
    p = (df.groupby(['controller', 'condition'])['success']
           .mean().unstack(fill_value=np.nan).reindex(CONTROLLERS))
    p.plot(kind='bar', ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Success rate')
    ax.set_title('Success rate per controller × condition')
    ax.legend(title='condition')
    return fig


def fig_time_violin(df):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    df2 = _clean(df)
    data, labels = [], []
    for ctrl in CONTROLLERS:
        for cond in CONDITIONS:
            s = df2[(df2.controller == ctrl) & (df2.condition == cond)]['time_to_goal'].values
            if len(s) > 0:
                data.append(s); labels.append(f'{ctrl}\n{cond}')
    if data:
        ax.violinplot(data, showmedians=True)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels)
    else:
        _no_data(ax)
    ax.set_ylabel('Time to goal (s)')
    ax.set_title('Time-to-goal distributions')
    return fig


def fig_efficiency_box(df):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    df2 = _clean(df)
    data, labels = [], []
    for ctrl in CONTROLLERS:
        for cond in CONDITIONS:
            s = df2[(df2.controller == ctrl) & (df2.condition == cond)]['path_efficiency'].values
            if len(s) > 0:
                data.append(s); labels.append(f'{ctrl}\n{cond}')
    if data:
        ax.boxplot(data)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels)
    else:
        _no_data(ax)
    ax.set_ylabel('Path efficiency (straight / actual)')
    ax.set_title('Path efficiency')
    return fig


def fig_clearance(df):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
    df2 = _clean(df)
    if df2.empty:
        _no_data(a1); _no_data(a2)
        a1.set_title('Min clearance'); a2.set_title('Close-approach counts')
        return fig
    cl_pivot = (df2.groupby(['controller', 'condition'])['min_clearance']
                  .median().unstack().reindex(CONTROLLERS))
    cl_pivot.plot(kind='bar', ax=a1)
    a1.set_ylabel('Median min clearance (m)')
    a1.set_title('Min clearance')
    a1.legend(title='condition')

    ca_pivot = (df.groupby(['controller', 'condition'])['n_close_approach']
                  .sum().unstack(fill_value=0).reindex(CONTROLLERS))
    ca_pivot.plot(kind='bar', ax=a2)
    a2.set_ylabel('Total close-approach samples')
    a2.set_title(f'Close-approach counts (<{CLOSE_APPROACH_THRESHOLD:.2f} m)')
    a2.legend(title='condition')
    return fig


def fig_jerk_degradation(df):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
    df2 = _clean(df)
    if df2.empty:
        _no_data(a1); _no_data(a2)
        a1.set_title('Mean jerk'); a2.set_title('Time-to-goal degradation')
        return fig
    j = (df2.groupby(['controller', 'condition'])['mean_jerk']
            .mean().unstack().reindex(CONTROLLERS))
    j.plot(kind='bar', ax=a1)
    a1.set_ylabel('Mean |jerk|')
    a1.set_title('Mean jerk')
    a1.legend(title='condition')

    t = (df2.groupby(['controller', 'condition'])['time_to_goal']
            .median().unstack().reindex(CONTROLLERS))
    if {'static', 'dynamic'}.issubset(t.columns):
        deg = (t['dynamic'] - t['static']) / t['static'].replace(0, np.nan)
        deg.plot(kind='bar', ax=a2, color='C3')
        a2.set_ylabel('(dynamic - static) / static')
        a2.set_title('Time-to-goal degradation: static → dynamic')
    else:
        _no_data(a2); a2.set_title('Time-to-goal degradation')
    return fig


_figs = {
    'success_rate':     fig_success_rate(df_all),
    'time_violin':      fig_time_violin(df_all),
    'efficiency_box':   fig_efficiency_box(df_all),
    'clearance':        fig_clearance(df_all),
    'jerk_degradation': fig_jerk_degradation(df_all),
}
for name, f in _figs.items():
    f.tight_layout()
    f.savefig(FIGURES_DIR / f'{name}.png', dpi=150)
    f.savefig(FIGURES_DIR / f'{name}.pdf')
plt.show()
print(f'Saved {len(_figs)} figures to {FIGURES_DIR}/')


In [ ]:
from itertools import combinations
from scipy import stats


def cliffs_delta(x, y) -> float:
    x = np.asarray(x); y = np.asarray(y)
    if x.size == 0 or y.size == 0:
        return float('nan')
    gt = int((x[:, None] > y[None, :]).sum())
    lt = int((x[:, None] < y[None, :]).sum())
    return (gt - lt) / (x.size * y.size)


def stats_dynamic_time(df: pd.DataFrame):
    if df.empty:
        print('No data loaded.')
        return None
    dyn = _clean(df[df.condition == 'dynamic'])
    groups = {c: dyn[dyn.controller == c]['time_to_goal'].values for c in CONTROLLERS}
    groups = {k: v for k, v in groups.items() if len(v) > 0}
    if len(groups) < 2:
        print('Need at least two controllers with dynamic data — skipping.')
        return None
    H, p_kw = stats.kruskal(*groups.values())
    print(f'Kruskal-Wallis on dynamic time-to-goal: H={H:.3f}, p={p_kw:.4f}')
    rows = []
    for a, b in combinations(groups.keys(), 2):
        u, p_mw = stats.mannwhitneyu(groups[a], groups[b],
                                     alternative='two-sided')
        d = cliffs_delta(groups[a], groups[b])
        rows.append({'A': a, 'B': b, 'U': u, 'p_value': p_mw,
                     'cliffs_delta': round(d, 3)})
    return pd.DataFrame(rows)


stats_dynamic_time(df_all)
